In [ ]:
# ===========================================================================
# UA-SPEECH MODEL TRAINING - interactive driver
#
# All training logic lives in src/training/; this notebook only calls it and
# stores results, matching notebooks/01_data_pipeline.ipynb's convention -
# functions and architecture belong in src/, only the act of running training
# and storing models happens here.
#
#   src/training/models.py     model factory: acoustic / deep_frozen / deep_lora / fusion
#   src/training/runner.py     TrainingConfig, run_training() - the fold loop
#   src/training/baseline.py   Phase 2: frozen wav2vec + linear SVM baseline
#   src/training/engine.py     one epoch: AMP, gradient clipping, optimizer
#   src/training/metrics.py    accuracy / precision / recall / specificity / F1 / AUROC
#   src/training/reporting.py  predictions / metrics / confusion-matrix / ROC / embeddings I/O
#
# See ROADMAP.md for the phase plan this notebook implements (Phase 1 sanity
# check, Phase 2 baseline reproduction and comparison).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("UA-Speech Training Notebook")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))
print_kv("Speakers", df_m6["Speaker_ID"].nunique())

In [ ]:
# STAGE 1 - Pipeline sanity check ("smoke test"). Trains the cheapest model
# (MFCC-only) for one fold, one epoch, on a tiny slice of data. This is NOT a
# real result - it exists to confirm the whole chain (model init, optimizer,
# scheduler, AMP, gradient clipping, early stopping, checkpointing,
# TensorBoard logging, prediction/metric/confusion-matrix/ROC/embedding
# writers) actually runs end to end before spending GPU time on a real run.
from src.training.runner import TrainingConfig, run_training

smoke_cfg = TrainingConfig(
    task="detection", model="acoustic",
    epochs=1, max_folds=1, limit_samples=24,
    run_name="_smoke_test",
)
run_training(df_m6, smoke_cfg)

In [ ]:
# STAGE 2 - Phase 2, step 1: reproduce the ICASSP base paper's feature
# extractor. Frozen wav2vec 2.0 (no LoRA, no fine-tuning) -> one 768-dim
# embedding per utterance PER HIDDEN-STATE LAYER (13 layers: the CNN
# feature-extractor output + 12 transformer layers). The base paper finds
# different layers win for different tasks (layer 1 for detection, layer 13
# for severity), so all 13 are extracted here rather than just the final
# layer - Stage 3 sweeps them to find which wins on this reproduction.
# Identical across every LOSO fold, so extracted once and cached to
# outputs/embeddings/.
from src.training.baseline import extract_frozen_embeddings_all_layers

frozen_embeddings_all_layers = extract_frozen_embeddings_all_layers(df_m6, batch_size=16)
print(f"Frozen embeddings (all layers): {frozen_embeddings_all_layers.shape}")

In [ ]:
# STAGE 3 - Phase 2, step 2: frozen wav2vec 2.0 -> linear SVM, swept across
# all 13 layers and evaluated on the full 28-fold LOSO detection protocol -
# exactly the base paper's pipeline and per-layer comparison. The paper's
# own reported result is layer 1 at 93.95% accuracy; the best layer found
# here is the number every other model in this project has to beat to be a
# genuine improvement, not an assumed one. If the best layer or accuracy
# lands far from the paper's, that's worth investigating (preprocessing,
# VAD, clip length) before trusting the ablation table in Stage 4.
from src.training.baseline import sweep_svm_baseline_layers

detection_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="detection", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

best_layer = int(detection_layer_sweep.iloc[0]["layer"])
baseline_pooled = detection_layer_sweep.iloc[0].to_dict()
baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Detection")
print_kv("Best layer", f"{best_layer} (paper reports layer 1 at 93.95% accuracy)")
print_kv("Accuracy", f"{baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{baseline_pooled['f1']:.4f}")
print_kv("Recall (sensitivity)", f"{baseline_pooled['recall']:.4f}")
print_kv("Precision", f"{baseline_pooled['precision']:.4f}")
print_kv("Specificity", f"{baseline_pooled['specificity']:.4f}")
print_kv("AUROC", f"{baseline_pooled['auroc']:.4f}")

In [ ]:
# STAGE 4 - Phase 2, step 3: the same frozen wav2vec 2.0 + linear SVM layer
# sweep, on the severity task's 81-fold balanced leave-one-per-class-out
# protocol (config.DROPPED_FOR_BALANCE, corrected to match the base paper's
# stated exclusion criterion - see src/config.py). The paper's own reported
# result is layer 13 (final) at 44.56% accuracy (4-class) - a low absolute
# number, expected for a 4-way severity task, but the target to compare
# against before trusting the severity ablation in Stage 7.
from src.training.baseline import sweep_svm_baseline_layers

severity_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="severity", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

severity_best_layer = int(severity_layer_sweep.iloc[0]["layer"])
severity_baseline_pooled = severity_layer_sweep.iloc[0].to_dict()
severity_baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Severity")
print_kv("Best layer", f"{severity_best_layer} (paper reports layer 13/final at 44.56% accuracy)")
print_kv("Accuracy", f"{severity_baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{severity_baseline_pooled['f1']:.4f}")

In [ ]:
# STAGE 5 - Phase 2 step 3 + Phase 3 ablation: full-scale training of all six
# variants - Frozen wav2vec + MLP, LoRA wav2vec + MLP, MFCC CNN, concatenated
# Fusion, and the two Phase 6 attention-fusion variants - on the complete
# 28-fold LOSO detection protocol, full epochs, no sample caps. This is the
# comparison the paper's contribution rests on: fusion -> attention_fusion ->
# attention_fusion_praat, same two pathways/data, only the fusion mechanism
# changes.
#
# NOTE ON SCALE: this is the real run, not a demo - a full 28-fold LOSO pass
# of a wav2vec-fine-tuning variant is hours of GPU time, not minutes, so all
# six variants is realistically a long unattended job. On a laptop GPU that
# is not safe to run continuously for hours/days unattended, so this cell
# is session-bounded: SESSION_BUDGET_HOURS caps how long *this call* trains
# for, and run_training() stops cleanly at the next fold boundary once the
# budget is used up. run_training() also skips any fold whose output already
# exists on disk (see src/training/runner.py), so simply re-running this
# cell later resumes from wherever the last session stopped - across folds
# AND across variants - instead of restarting.
from src.training.runner import TrainingConfig, run_training
from src.training.models import MODEL_NAMES
from src.console import print_note
import time

SESSION_BUDGET_HOURS = 2.5  # lower for a supervised first session, raise once you know real fold timing
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

comparison_pooled = {"baseline_svm": baseline_pooled}

for model_name in MODEL_NAMES:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        comparison_pooled[model_name] = pooled

In [ ]:
# STAGE 6 - Phase 2/3 comparison table: baseline SVM vs. every trained variant,
# pooled metrics side by side. Saved to outputs/metrics/phase2_comparison.csv
# for the paper/report.
comparison_df = pd.DataFrame(comparison_pooled).T
comparison_df.index.name = "model"

comparison_path = config.METRICS_DIR / "phase2_comparison.csv"
comparison_df.to_csv(comparison_path)

print_header("Phase 2/3 Comparison - Detection")
print_kv("Saved to", comparison_path)
comparison_df

In [ ]:
# STAGE 6.5 - Pick which variants go on to the severity task. Stage 7 below
# runs the 81-fold severity protocol (leave-one-speaker-per-class-out), which
# is ~3x the fold count of detection's 28-fold LOSO - across all six variants
# that's the single largest chunk of total GPU time in this notebook. Only
# the architectures that actually won the detection comparison matter for
# the severity story, so rank Stage 5's results by F1 and carry forward just
# the top TOP_K_FOR_SEVERITY into Stage 7 instead of all six.
TOP_K_FOR_SEVERITY = 2

ranked_variants = sorted(
    (m for m in MODEL_NAMES if m in comparison_pooled),
    key=lambda m: comparison_pooled[m]["f1"], reverse=True,
)
severity_model_names = ranked_variants[:TOP_K_FOR_SEVERITY]

print_note(f"Running severity (81-fold) only for top {TOP_K_FOR_SEVERITY} "
          f"detection variants by F1: {severity_model_names}")

In [ ]:
# STAGE 7 - Severity task: the top variants from Stage 6.5 (by default, the
# best TOP_K_FOR_SEVERITY of the six by detection F1 - see the cell above) on
# the balanced leave-one-speaker-per-class-out protocol (81 folds; see
# src/splits.py and config.DROPPED_FOR_BALANCE), plus the Stage 4 severity
# SVM baseline for a like-for-like comparison table (mirrors Stage 5/6's
# detection pattern).
#
# Lower priority than Stage 5's detection run (the paper's primary
# comparison) - run this once Stage 5 is done or far enough along, since
# 81 folds is a larger job than 28 even before multiplying by variant count.
# Same session-bounded pattern as Stage 5: SESSION_BUDGET_HOURS caps this
# call, run_training() stops cleanly at a fold boundary, and re-running the
# cell resumes rather than restarts.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

SESSION_BUDGET_HOURS = 2.5  # lower for a supervised first session, raise once you know real fold timing
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

severity_pooled = {"baseline_svm": severity_baseline_pooled}

for model_name in severity_model_names:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

severity_df = pd.DataFrame(severity_pooled).T
severity_df.index.name = "model"

severity_path = config.METRICS_DIR / "phase3_severity_comparison.csv"
severity_df.to_csv(severity_path)

print_header("Phase 3 Comparison - Severity")
print_kv("Saved to", severity_path)
severity_df